# 23_representation_ensemble — 표현 기반 앙상블 (신규, 재설계)

**한 줄 요약:** 지문 6종·2D·3D descriptor를 **각각 독립 표현**으로 두고, **(모델 5종 × 표현 8종)=40개 조합**을 전부 평가해 **성능 좋은 조합만** 골라 소프트보팅 앙상블을 만든다.
**이전(21)과 차이:** 21은 전 특징을 한 덩어리로 concat해 알고리즘끼리만 앙상블 → 지문별 기여를 못 봄. 여기선 **표현마다 따로 학습**해 어떤 (모델×표현)이 좋은지 드러나고 앙상블 다양성↑.
**불균형 처리:** active를 하나도 안 버리고 `class_weight='balanced'`(XGB는 scale_pos_weight)로 보정.
**정직성 점검:** test를 active/decoy/real_inactive로 쪼개 실제 실력을 본다.
**근거:** Wolpert(스태킹)·Willett(data fusion)·Chen 2019(decoy bias)·Bahia 2023(2D/3D). → `docs/references_modeling.md`
**큰 흐름:** ① 준비 → ② 데이터 → ③ 표현정의 → ④ 모델정의 → ⑤ 40조합 격자평가 → ⑥ 상위조합 앙상블 → ⑦ source별 정직성 점검 → ⑧ 저장

> **📌 읽는 법**: 각 코드 셀은 [① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기].

### 준비 — 도구 불러오기
5개 알고리즘·전처리·평가지표를 가져온다.

In [ ]:
import os
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')
print('작업 폴더:', os.getcwd())
import numpy as np, pandas as pd, pickle, time
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, matthews_corrcoef
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

🔎 **코드 뜯어보기 (준비)**
- `scale_pos_weight`용 XGBClassifier, `class_weight`용 나머지. `SimpleImputer`=결측(3D NaN) 채움, `StandardScaler`=LogReg용 표준화.

### 셀 1 — 데이터 로드
22가 만든 재균형 멤버십(어떤 분자·어느 split)에 v2 특징을 붙인다.

In [ ]:
# 특징표(v2) + 재균형 멤버십(22) 로드 → 재균형에 뽑힌 분자만, 지정된 split으로
V2  = "data/HSD17B13_final_training_1to1_v2.csv"
MEM = "data/HSD17B13_rebalanced_membership.csv"
feat = pd.read_csv(V2)                                   # 지문+2D+3D+potency
mem  = pd.read_csv(MEM)                                  # canonical_smiles, source, potency, split
df   = mem.merge(feat.drop(columns=["potency"]), on="canonical_smiles", how="left")
print("재균형 학습표:", df.shape, "| split:", dict(df.split.value_counts()))
print("불균형 비율(train) active:inactive =",
      dict(df[df.split=='train'].potency.value_counts()))

🔎 **코드 뜯어보기 (셀 1)**
- `mem.merge(feat.drop(columns=['potency']), on='canonical_smiles')` : 멤버십(분자·split)에 특징표를 붙임. potency는 멤버십 것을 사용(중복 제거).

### 셀 2 — 표현(representation) 정의
지문 6종 + 2D + 3D를 각각 '열 묶음'으로 정의한다(이게 base 모델들이 각자 볼 입력).

In [ ]:
# '표현(representation)' 정의 — 지문 6종 + 2D descriptor + 3D descriptor를 각각 독립 표현으로
fp_pref = ["ecfp4_", "rdkit_", "atompair_", "topotorsion_", "maccs_", "avalon_"]
def cols_with(pref): return [c for c in df.columns if c.startswith(pref)]
meta = {"canonical_smiles", "source", "potency", "split"}
d3_cols = [c for c in df.columns if c.startswith("d3_")]
fp_all  = [c for p in fp_pref for c in cols_with(p)]
d2_cols = [c for c in df.columns if c not in meta and c not in fp_all and c not in d3_cols]

REPS = {p.rstrip("_"): cols_with(p) for p in fp_pref}    # 지문 6종
REPS["desc2d"] = d2_cols                                 # 2D descriptor(WEKA 선택본)
REPS["desc3d"] = d3_cols                                 # 3D descriptor
print("표현 목록(열 수):", {k: len(v) for k, v in REPS.items()})

🔎 **코드 뜯어보기 (셀 2)**
- `cols_with(pref)` : 접두사로 그 지문의 열만 모음. `REPS`=표현이름→열목록 딕셔너리. desc2d는 지문·3D·메타를 뺀 나머지.

### 셀 3 — 알고리즘 정의 (불균형 보정)
5개 모델 전부 class_weight로 소수 클래스(inactive)를 더 무겁게 취급하게 만든다.

In [ ]:
# 알고리즘 5종 — 전부 class_weight로 불균형 보정(active를 안 버리는 핵심)
n_pos = int((df[df.split=='train'].potency == 1).sum())   # active(다수)
n_neg = int((df[df.split=='train'].potency == 0).sum())   # inactive(소수)
spw = n_neg / n_pos                                       # XGBoost용: 다수(양성) 가중치 축소
def models():
    imp = lambda: SimpleImputer(strategy="median")        # 3D의 NaN(대형분자) 채움
    return {
        "RF":  Pipeline([("i", imp()), ("m", RandomForestClassifier(
                    n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1))]),
        "ET":  Pipeline([("i", imp()), ("m", ExtraTreesClassifier(
                    n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1))]),
        "LGBM":Pipeline([("i", imp()), ("m", LGBMClassifier(
                    n_estimators=400, class_weight="balanced", random_state=42, n_jobs=-1, verbosity=-1))]),
        "XGB": Pipeline([("i", imp()), ("m", XGBClassifier(
                    n_estimators=400, scale_pos_weight=spw, random_state=42, n_jobs=-1,
                    eval_metric="logloss", verbosity=0))]),
        "LR":  Pipeline([("i", imp()), ("s", StandardScaler()), ("m", LogisticRegression(
                    max_iter=3000, class_weight="balanced"))]),
    }
print(f"train 양성(active) {n_pos} : 음성(inactive) {n_neg} | XGB scale_pos_weight={spw:.3f}")

def xy(cols, part):
    m = df.split == part
    return df.loc[m, cols].to_numpy(), df.loc[m, "potency"].to_numpy()

def mcc_of(y, proba):
    return matthews_corrcoef(y, (proba >= 0.5).astype(int))

🔎 **코드 뜯어보기 (셀 3)**
- `class_weight='balanced'` : 클래스 빈도의 역수로 자동 가중 → 소수(inactive) 오분류에 큰 벌점. active를 안 버리고 불균형을 다루는 핵심.
- `scale_pos_weight = n_neg/n_pos` : XGBoost엔 class_weight가 없어 이 값으로 다수(양성=active)를 상대적으로 낮춤.
- `xy(cols, part)` : 지정 split·지정 표현의 (X, y)를 numpy로 반환.

### 셀 4 — (모델×표현) 40조합 격자 평가
모든 조합을 학습해 검증 MCC 표를 만든다. 여기서 어떤 조합이 좋은지 한눈에.

In [ ]:
# (모델 × 표현) 40개 조합을 전부 학습 → 검증(val) MCC로 성능표. 여기서 좋은 조합을 고른다
t0 = time.time()
grid, fitted = [], {}
for rn, cols in REPS.items():
    Xtr, ytr = xy(cols, "train"); Xva, yva = xy(cols, "val")
    for mn, model in models().items():
        model.fit(Xtr, ytr)
        vmcc = mcc_of(yva, model.predict_proba(Xva)[:, 1])
        grid.append({"model": mn, "rep": rn, "val_MCC": round(vmcc, 3)})
        fitted[(mn, rn)] = model
G = pd.DataFrame(grid)
piv = G.pivot(index="rep", columns="model", values="val_MCC")
print(f"검증 MCC 표 (행=표현, 열=모델)  [{time.time()-t0:.0f}s]")
print(piv.to_string())
print("\n조합 상위 8 (val_MCC):")
print(G.sort_values("val_MCC", ascending=False).head(8).to_string(index=False))

🔎 **코드 뜯어보기 (셀 4)**
- 이중 for문으로 표현×모델 전부 학습. `G.pivot(...)`=행=표현/열=모델 격자표. `fitted[(mn,rn)]`=학습된 파이프라인 보관(앙상블에 재사용).

### 셀 5 — 상위 조합 소프트보팅 앙상블
검증 MCC 상위 K개 조합의 예측확률을 평균낸다(각 base는 자기 표현만 사용). 단일 최고와도 비교.

In [ ]:
# 상위 K개 (모델×표현) 조합만 골라 소프트보팅 앙상블(각 base는 자기 표현만 사용)
K = 5
top = G.sort_values("val_MCC", ascending=False).head(K)[["model", "rep"]].values.tolist()
print("앙상블 base(상위", K, "조합):", top)

def metrics(y, proba):
    pred = (proba >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    return dict(MCC=matthews_corrcoef(y, pred), ROC=roc_auc_score(y, proba),
                PR=average_precision_score(y, proba), Acc=(tp+tn)/(tp+tn+fp+fn),
                Recall=tp/(tp+fn) if (tp+fn) else 0, Prec=tp/(tp+fp) if (tp+fp) else 0)

def ens_proba(part):                                     # 상위 조합들의 예측확률 평균(소프트보팅)
    ps = []
    for mn, rn in top:
        Xp, _ = xy(REPS[rn], part)
        ps.append(fitted[(mn, rn)].predict_proba(Xp)[:, 1])
    return np.mean(ps, axis=0)

yte = df.loc[df.split=="test", "potency"].to_numpy()
p_ens = ens_proba("test")
print("\n[표현 앙상블] test 전체:", {k: round(v,3) for k,v in metrics(yte, p_ens).items()})
# 단일 최고 조합과 비교
bmn, brn = top[0]
Xte1,_ = xy(REPS[brn], "test"); p_best = fitted[(bmn,brn)].predict_proba(Xte1)[:,1]
print(f"[단일 최고={bmn}×{brn}] test 전체:", {k: round(v,3) for k,v in metrics(yte, p_best).items()})

🔎 **코드 뜯어보기 (셀 5)**
- `top = G.sort_values('val_MCC').head(K)` : 검증 성능 상위 K조합 선정.
- `ens_proba(part)` : 각 조합이 **자기 표현의 열만** 뽑아 예측한 확률들을 `np.mean`으로 평균(소프트보팅). 표현이 달라 다양성↑.

### 셀 6 — ★ source별 정직성 점검
test를 active/decoy/real_inactive로 나눠 실제 실력을 본다. 예전 concat 단일모델과도 비교.

In [ ]:
# ★ 정직성 점검: test를 source별(active/decoy/real_inactive)로 나눠 얼마나 맞히나
src_te = df.loc[df.split=="test", "source"].to_numpy()
def by_source(proba, tag):
    pred = (proba >= 0.5).astype(int)
    print(f"\n[{tag}] source별 정답률:")
    for s in ["active", "decoy", "real_inactive"]:
        m = src_te == s
        if s == "active":
            print(f"  {s:14s} n={m.sum():3d} | active로 맞힘(recall) {(pred[m]==1).mean():.3f} | 평균확률 {proba[m].mean():.3f}")
        else:
            print(f"  {s:14s} n={m.sum():3d} | inactive로 맞힘        {(pred[m]==0).mean():.3f} | 평균확률 {proba[m].mean():.3f}")
by_source(p_ens, "표현 앙상블")

# 비교군: 예전 방식(전 특징 concat) 단일 LightGBM (같은 재균형 데이터·split)
Xtr,ytr = xy(fp_all + d2_cols + d3_cols, "train")
Xte,_   = xy(fp_all + d2_cols + d3_cols, "test")
base_lgbm = Pipeline([("i", SimpleImputer(strategy="median")),
                      ("m", LGBMClassifier(n_estimators=400, class_weight="balanced",
                                           random_state=42, n_jobs=-1, verbosity=-1))]).fit(Xtr, ytr)
p_concat = base_lgbm.predict_proba(Xte)[:, 1]
print("\n[참고: 전특징 concat 단일 LGBM] test 전체:", {k: round(v,3) for k,v in metrics(yte, p_concat).items()})
by_source(p_concat, "전특징 concat LGBM")

🔎 **코드 뜯어보기 (셀 6)**
- `by_source(proba, tag)` : source별로 active는 recall(활성 맞힘), inactive는 정답(비활성 맞힘) 비율. **real_inactive 점수가 진짜 실력에 가까움.**
- 마지막에 '전 특징 concat 단일 LGBM'을 같은 데이터로 돌려 방식 차이를 비교.

### 셀 7 — 저장
앙상블(상위 조합의 학습된 파이프라인)·격자표·test 요약을 저장한다.

In [ ]:
# 저장: 앙상블(상위 조합의 표현·학습된 파이프라인) + 성능표
bundle = {"top": top, "reps": REPS,
          "pipelines": {f"{mn}|{rn}": fitted[(mn, rn)] for mn, rn in top}}
with open("data/HSD17B13_repr_ensemble.pkl", "wb") as f:
    pickle.dump(bundle, f)
G.to_csv("data/HSD17B13_repr_grid.csv", index=False)
# test 요약표(앙상블 vs 단일최고 vs concat)
summ = pd.DataFrame({
    "표현앙상블": metrics(yte, p_ens),
    "단일최고":   metrics(yte, p_best),
    "concat단일": metrics(yte, p_concat),
}).T.round(3)
summ.to_csv("data/HSD17B13_repr_test_summary.csv")
print("\n=== test 요약(재균형·class_weight 적용) ===")
print(summ.to_string())
print("\n저장: data/HSD17B13_repr_ensemble.pkl, HSD17B13_repr_grid.csv, HSD17B13_repr_test_summary.csv")

🔎 **코드 뜯어보기 (셀 7)**
- `pickle.dump(bundle)` : 상위 조합의 표현·파이프라인을 저장(재사용). `summ`=앙상블/단일최고/concat의 test 지표 비교표.